In [ ]:
import os
import requests
import logging
from requests.exceptions import RequestException
import json

USERNAME = "axone"
PASSWORD = "Ax0nesys!"
SERVER_URL = "http://192.168.20.1"  
VERIFY_CERTIFICATES = os.getenv('VERIFY_CERTIFICATES', 'True') == 'True'

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def get_access_token(session):
    """Retrieve access token from the Identity Provider (IDP)."""
    try:
        response = session.post(
            f"{SERVER_URL}/IDP/connect/token",
            headers={'Content-Type': 'application/x-www-form-urlencoded'},
            data={
                'grant_type': 'password',
                'username': USERNAME,
                'password': PASSWORD,
                'client_id': 'GrantValidatorClient'
            },
            verify=VERIFY_CERTIFICATES
        )
        response.raise_for_status()
        token = response.json().get('access_token')
        if not token:
            logger.error("Access token not found in the response.")
            return None
        logger.info("Access token retrieved successfully.")
        
        # Save the token to a text file
        with open('access_token.txt', 'w') as file:
            file.write(token)
        logger.info("Access token saved to 'access_token.txt'.")
        
        return token
    except RequestException as e:
        logger.error(f"Error retrieving token: {e}")
        return None

def fetch_data(session, token, endpoint, description):
    """Generic function to fetch data from a given endpoint."""
    url = f"{SERVER_URL}{endpoint}"
    try:
        print(f"Fetching data from {url}")
        response = session.get(
            url,
            headers={'Authorization': f'Bearer {token}'},
            verify=VERIFY_CERTIFICATES
        )
        print(f"Status Code: {response.status_code}")
        response.raise_for_status()
        data = response.json()
        print(f"Response Data for {description}: {data}")
        logger.info(f"Retrieved {len(data)} {description}.")
        return data
    except RequestException as e:
        logger.error(f"Error retrieving {description}: {e}")
        return None

def fetch_hardware_driver_settings(session, token, hardware_id):
    """Fetch hardware driver settings for a given hardware ID."""
    endpoint = f"/api/rest/v1/hardwareDriverSettings/{hardware_id}?definitions"
    url = f"{SERVER_URL}{endpoint}"
    try:
        print(f"Fetching hardware driver settings from {url}")
        response = session.get(
            url,
            headers={'Authorization': f'Bearer {token}'},
            verify=VERIFY_CERTIFICATES
        )
        print(f"Status Code: {response.status_code}")
        response.raise_for_status()
        settings = response.json()
        print(f"Driver Settings for hardware ID {hardware_id}: {settings}")
        logger.info(f"Retrieved driver settings for hardware ID {hardware_id}.")
        return settings
    except RequestException as e:
        logger.error(f"Error retrieving driver settings for hardware ID {hardware_id}: {e}")
        return None

def save_to_json(data, filename):
    """Save data to a JSON file."""
    try:
        with open(filename, 'w') as f:
            json.dump(data, f, indent=4)
        logger.info(f"Data saved to {filename}.")
    except IOError as e:
        logger.error(f"Error saving data to {filename}: {e}")

def main():
    """Main function to retrieve token and fetch various data."""
    if not all([USERNAME, PASSWORD]):
        logger.error("API_USERNAME and API_PASSWORD must be set as environment variables.")
        return

    with requests.Session() as session:
        token = get_access_token(session)
        if not token:
            return

        # Fetch and save each general endpoint data
        endpoints = {
            '/api/rest/v1/events': 'events',
            '/api/rest/v1/cameras': 'cameras',
            '/api/rest/v1/hardware': 'hardware',
            '/api/rest/v1/sites': 'sites',
            '/api/rest/v1/recordingServers': 'recordingServers',
            '/api/rest/v1/alarms': 'alarms',
            '/api/ws/messages/v1':'messages',
            '/api/rest/v1/alarmStates': 'alarmStates',
            '/api/rest/v1/alarmPriorities': 'alarmPriorities',
            '/api/rest/v1/alarms?page=0&size=10':'last10',
            '/api/ws/events/v1':'ezmone',
            '/api/rest/v1/cameras/54fa3c64-ab8d-4455-aa08-35ba6b552228': 'uniq'  # Added care plans endpoint

            
        }

        for endpoint, description in endpoints.items():
            data = fetch_data(session, token, endpoint, description)
            if data is not None:
                filename = f"{description}.json"
                save_to_json(data, filename)
                hardware_ids = [item.get('id') for item in data if 'id' in item]
                print(f"Extracted hardware IDs: {hardware_ids}")
                logger.info(f"Extracted hardware IDs: {hardware_ids}")

                hardware_settings = []
                for hardware_id in hardware_ids:
                    settings = fetch_hardware_driver_settings(session, token, hardware_id)
                    if settings:
                        hardware_settings.append({'hardware_id': hardware_id, 'driverSettings': settings})

                # Save hardware driver settings data to a separate JSON
                save_to_json(hardware_settings, "hardware_driver_settings.json")

if __name__ == '__main__':
    main()


INFO:__main__:Access token retrieved successfully.
INFO:__main__:Access token saved to 'access_token.txt'.
INFO:__main__:Fetching page 0 from /api/rest/v1/alarms?page=0&size=100
INFO:__main__:Fetching page 1 from /api/rest/v1/alarms?page=1&size=100
INFO:__main__:Fetching page 2 from /api/rest/v1/alarms?page=2&size=100
INFO:__main__:Fetching page 3 from /api/rest/v1/alarms?page=3&size=100
INFO:__main__:Fetching page 4 from /api/rest/v1/alarms?page=4&size=100
INFO:__main__:Fetching page 5 from /api/rest/v1/alarms?page=5&size=100
INFO:__main__:Fetching page 6 from /api/rest/v1/alarms?page=6&size=100
INFO:__main__:Fetching page 7 from /api/rest/v1/alarms?page=7&size=100
INFO:__main__:Fetching page 8 from /api/rest/v1/alarms?page=8&size=100
INFO:__main__:Fetching page 9 from /api/rest/v1/alarms?page=9&size=100
INFO:__main__:Fetching page 10 from /api/rest/v1/alarms?page=10&size=100
INFO:__main__:Fetching page 11 from /api/rest/v1/alarms?page=11&size=100
INFO:__main__:Fetching page 12 from /

Fetching data from http://192.168.20.1/api/rest/v1/events


INFO:__main__:Retrieved 2 events.
INFO:__main__:Data saved to events.json.


Status Code: 200
Response Data for events: {'array': [{'specversion': '1.0', 'type': '023d4d8d-5848-43cf-8f94-4f7f230966ef', 'source': 'recordingServers/27ed6e15-babf-4c4f-a86e-cf3d2a188ec1', 'time': '2025-01-05T15:38:37.118709Z', 'id': 'ff8d753b-048b-46cc-80e6-a39d7a02553e', 'datatype': 'none'}, {'specversion': '1.0', 'type': '023d4d8d-5848-43cf-8f94-4f7f230966ef', 'source': 'recordingServers/27ed6e15-babf-4c4f-a86e-cf3d2a188ec1', 'time': '2025-01-05T15:47:47.0960656Z', 'id': '3ca07b6e-e3e7-485c-bcb2-919e7c7b62b1', 'datatype': 'none'}, {'specversion': '1.0', 'type': '023d4d8d-5848-43cf-8f94-4f7f230966ef', 'source': 'recordingServers/27ed6e15-babf-4c4f-a86e-cf3d2a188ec1', 'time': '2025-01-05T15:49:07.1005142Z', 'id': '50cae4b6-2d0b-43bb-baa5-0e87fa1cfa10', 'datatype': 'none'}, {'specversion': '1.0', 'type': '023d4d8d-5848-43cf-8f94-4f7f230966ef', 'source': 'recordingServers/27ed6e15-babf-4c4f-a86e-cf3d2a188ec1', 'time': '2025-01-05T15:50:17.1040633Z', 'id': 'd18d6d66-a1c1-42fc-afea-045

INFO:__main__:Retrieved 1 cameras.
INFO:__main__:Data saved to cameras.json.


Status Code: 200
Response Data for cameras: {'array': [{'displayName': 'BX520-HD | Showroom', 'enabled': True, 'id': '19bab11f-b0f3-47fb-b6f4-891de3ebe953', 'name': 'BX520-HD | Showroom', 'channel': 0, 'description': '', 'createdDate': '0001-01-01T00:00:00.0000000', 'lastModified': '2024-04-26T09:39:07.8700000Z', 'gisPoint': 'POINT EMPTY', 'shortName': '', 'icon': 0, 'coverageDirection': '0', 'coverageDepth': '0', 'coverageFieldOfView': '0', 'recordingFramerate': '5', 'recordKeyframesOnly': False, 'recordOnRelatedDevices': True, 'recordingEnabled': True, 'prebufferEnabled': True, 'prebufferInMemory': True, 'prebufferSeconds': 3, 'edgeStorageEnabled': False, 'edgeStoragePlaybackEnabled': False, 'manualRecordingTimeoutEnabled': True, 'manualRecordingTimeoutMinutes': 5, 'recordingStorage': {'type': 'storages', 'id': '7b95a077-cad3-4eaa-a79c-19177fb47425'}, 'relations': {'parent': {'type': 'hardware', 'id': 'd731a8d7-03d3-40e0-a638-a0d54aa8077d'}, 'self': {'type': 'cameras', 'id': '19bab11

INFO:__main__:Retrieved 1 hardware.
INFO:__main__:Data saved to hardware.json.
INFO:__main__:Retrieved 1 sites.
INFO:__main__:Data saved to sites.json.
INFO:__main__:Retrieved 1 recordingServers.
INFO:__main__:Data saved to recordingServers.json.


Status Code: 200
Response Data for hardware: {'array': [{'displayName': 'FD836BA (192.168.20.112)', 'enabled': True, 'id': '285ff8d3-1a24-4d70-af4d-a8c55b427bfc', 'name': 'FD836BA (192.168.20.112)', 'description': '', 'address': 'http://192.168.20.112/', 'userName': 'root', 'model': 'VIVOTEK FD836BA-HTV', 'passwordLastModified': '0001-01-01T00:00:00.0000000', 'lastModified': '2024-02-16T15:19:47.3630000Z', 'hardwareDriverPath': {'type': 'hardwareDrivers', 'id': '131480b9-2537-4f2c-a6d5-dd318b72264b'}, 'relations': {'parent': {'type': 'recordingServers', 'id': '27ed6e15-babf-4c4f-a86e-cf3d2a188ec1'}, 'self': {'type': 'hardware', 'id': '285ff8d3-1a24-4d70-af4d-a8c55b427bfc'}}}, {'displayName': 'FD836B (192.168.20.107)', 'enabled': True, 'id': '2d92c667-6cbd-43d0-9ba6-d1da411c80da', 'name': 'FD836B (192.168.20.107)', 'description': '', 'address': 'http://192.168.20.107/', 'userName': 'root', 'model': 'VIVOTEK FD836B-HTV', 'passwordLastModified': '0001-01-01T00:00:00.0000000', 'lastModifie

INFO:__main__:Access token retrieved successfully.
INFO:__main__:Access token saved to 'access_token.txt'.
INFO:__main__:Fetching page 0 from /api/rest/v1/alarms?page=0&size=100
INFO:__main__:Fetching page 1 from /api/rest/v1/alarms?page=1&size=100
INFO:__main__:Fetching page 2 from /api/rest/v1/alarms?page=2&size=100
INFO:__main__:Fetching page 3 from /api/rest/v1/alarms?page=3&size=100
INFO:__main__:Fetching page 4 from /api/rest/v1/alarms?page=4&size=100
INFO:__main__:Fetching page 5 from /api/rest/v1/alarms?page=5&size=100
INFO:__main__:Fetching page 6 from /api/rest/v1/alarms?page=6&size=100
INFO:__main__:Fetching page 7 from /api/rest/v1/alarms?page=7&size=100
INFO:__main__:Fetching page 8 from /api/rest/v1/alarms?page=8&size=100
INFO:__main__:Fetching page 9 from /api/rest/v1/alarms?page=9&size=100
INFO:__main__:Fetching page 10 from /api/rest/v1/alarms?page=10&size=100
INFO:__main__:Fetching page 11 from /api/rest/v1/alarms?page=11&size=100
INFO:__main__:Fetching page 12 from /

In [6]:
import json
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def load_json_file(filename):
    """Load JSON data from a file."""
    try:
        with open(filename, 'r') as file:
            return json.load(file).get('array', [])
    except FileNotFoundError as e:
        logger.error(f"File not found: {filename}")
        return []
    except json.JSONDecodeError as e:
        logger.error(f"Error decoding JSON in file {filename}: {e}")
        return []

def extract_camera_info_from_camera_json(camera_data):
    """Extract camera names and ids from camera data."""
    cameras = {}
    for camera in camera_data:
        camera_id = camera.get('id')
        name = camera.get('name')
        ip = camera.get('address')  # Assuming camera json has 'address' as the IP
        if camera_id and name and ip:
            cameras[camera_id] = {'name': name, 'ip': ip}
        else:
            logger.warning(f"Skipping camera with missing fields: {camera}")
    return cameras

def extract_camera_info_from_hardware(hardware_data, camera_ids):
    """Extract camera name and IP from hardware data, matching with camera ids."""
    camera_info = []
    
    for hardware in hardware_data:
        hardware_id = hardware.get('id')
        name = hardware.get('name')
        address = hardware.get('address')
        
        # Check if this hardware matches a camera by its ID
        if hardware_id in camera_ids and "http://" in address:
            ip_address = address.split('://')[1].split('/')[0]
            camera_info.append({'name': name, 'ip': ip_address})
            logger.info(f"Found matching camera: {name} with IP: {ip_address}")
        else:
            logger.warning(f"Skipping non-matching or non-camera hardware: {name}")
    
    return camera_info

def save_camera_info_to_json(camera_info, filename="ipcameras.json"):
    """Save extracted camera information to a JSON file."""
    try:
        with open(filename, 'w') as file:
            json.dump(camera_info, file, indent=4)
        logger.info(f"Camera information saved to {filename}.")
    except IOError as e:
        logger.error(f"Error saving camera information to JSON: {e}")

def main():
    """Main function to load data, cross-reference, and save filtered camera info."""
    # Load the camera data from camera.json
    camera_data = load_json_file('camera.json')
    
    # Extract camera info from camera.json (we are using 'id' for matching)
    camera_ids = extract_camera_info_from_camera_json(camera_data)

    if not camera_ids:
        logger.warning("No cameras found in camera.json.")
        return
    
    # Load the hardware data from hardware.json
    hardware_data = load_json_file('hardware.json')

    if not hardware_data:
        logger.warning("No hardware data found in hardware.json.")
        return

    # Extract only the matching cameras from hardware data
    matching_camera_info = extract_camera_info_from_hardware(hardware_data, camera_ids)

    if matching_camera_info:
        # Save the matched camera information to ipcameras.json
        save_camera_info_to_json(matching_camera_info)
    else:
        logger.warning("No matching cameras found.")

if __name__ == '__main__':
    main()


ERROR:__main__:File not found: camera.json


In [7]:
import requests
import logging
import json

# Configuration
SERVER_URL = "http://192.168.20.1"  # Replace with your server URL
TOKEN = "eyJhbGciOiJSUzI1NiIsImtpZCI6IkIxMUNDMDE5RjQ2MEE4OTBFMkVGQTQ2RkMwOEM2QjI0IiwidHlwIjoiSldUIn0.eyJpc3MiOiJodHRwOi8vc3J2LW1pbGVzdG9uZS5hZC5heG9uZS9JRFAiLCJuYmYiOjE3MzA5ODMwMTksImlhdCI6MTczMDk4MzAxOSwiZXhwIjoxNzMwOTg2NjE5LCJhdWQiOlsibWFuYWdlbWVudHNlcnZlciIsImh0dHA6Ly9zcnYtbWlsZXN0b25lLmFkLmF4b25lL0lEUC9yZXNvdXJjZXMiXSwic2NvcGUiOlsibWFuYWdlbWVudHNlcnZlciJdLCJhbXIiOlsicHdkIl0sImNsaWVudF9pZCI6IkdyYW50VmFsaWRhdG9yQ2xpZW50Iiwic3ViIjoiMTM4RkZFRkMtNEExRC00RUY3LTk3NDYtNDYzN0ZEMTRDQkIxIiwiYXV0aF90aW1lIjoxNzMwOTgzMDE5LCJpZHAiOiJsb2NhbCIsIm5hbWUiOiJBeG9uZSIsInVwZGF0ZWRfYXQiOiIxNzMwOTEwNTQyIiwianRpIjoiQjgxRjREOEVDNjFCRTQxNDEwRDM4MTA2NzBGMzUwOTcifQ.4sEWgjCCrSrv4J6150JYQk2rxLO-6rBug2xJWjco26plYHuYiKgtwFA9mlJW8oxjGx60XuotSsqTJHiqY17SYwJI7rcYrrSvcvfcDD8Qr5sUqCCuYNfKVtdn2GEs_DM3UIQ-ZxfN2nsXOWENT5dzjaeX1i4uw8c9Mun9w-5zNa1Ltd9RAClRNxgesmM_m_0og1pGGGR6fqkyraaV1HYKuJA908gkQF1-vRMRdPM7Dt5Md2aa_uJlF61M8jjjbnTU5_papK6iuoFSPisvefvNL9rbIB80Bam9IeFdg3EyxQRbVZT44pJSOutK5Fznvr3YmlsKUAyhJlp8yR7EV0aL5A"  # Replace with your access token
CAMERA_ID = "dfebfa67-3a9d-4e79-aa1b-2789dec1d844"  # Replace with the camera ID you want to check

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def get_camera_ip(camera_id):
    """Fetch the camera details and retrieve the IP address from the related hardware."""
    try:
        # Step 1: Fetch camera details
        camera_url = f"{SERVER_URL}/api/rest/v1/cameras/{camera_id}"
        camera_response = requests.get(
            camera_url,
            headers={'Authorization': f'Bearer {TOKEN}'}
        )
        camera_response.raise_for_status()
        camera_data = camera_response.json()
        
        # Log the full camera data to inspect the structure
        logger.info(f"Camera Data for {camera_id}: {json.dumps(camera_data, indent=4)}")

        # Step 2: Check for a 'relations' section that could link to hardware
        if 'relations' in camera_data.get('data', {}):
            camera_relations = camera_data['data']['relations']
            logger.info(f"Camera Relations: {json.dumps(camera_relations, indent=4)}")
            
            if 'parent' in camera_relations:
                # The parent relation may point to the hardware id
                hardware_id = camera_relations['parent'].get('id')
                if hardware_id:
                    logger.info(f"Found related hardware ID: {hardware_id}")
                    
                    # Step 3: Fetch the related hardware data
                    hardware_url = f"{SERVER_URL}/api/rest/v1/hardware/{hardware_id}"
                    hardware_response = requests.get(
                        hardware_url,
                        headers={'Authorization': f'Bearer {TOKEN}'}
                    )
                    hardware_response.raise_for_status()
                    hardware_data = hardware_response.json()
                    
                    logger.info(f"Hardware Data for {hardware_id}: {json.dumps(hardware_data, indent=4)}")
                    
                    # Extract the IP address from the hardware data
                    ip_address = hardware_data.get('data', {}).get('address')
                    if ip_address:
                        logger.info(f"Found IP address for camera {camera_id}: {ip_address}")
                        return ip_address
                    else:
                        logger.error(f"No IP address found for hardware {hardware_id}.")
                        return None
                else:
                    logger.error(f"No hardware ID found in camera relations.")
                    return None
            else:
                logger.error(f"No 'parent' relation found for camera {camera_id}.")
                return None
        else:
            logger.error(f"No relations found for camera {camera_id}.")
            return None

    except requests.exceptions.RequestException as e:
        logger.error(f"Error retrieving camera IP: {e}")
        return None


def main():
    """Main function to retrieve and display the IP address of a camera."""
    ip_address = get_camera_ip(CAMERA_ID)
    if ip_address:
        print(f"The IP address of camera {CAMERA_ID} is: {ip_address}")
    else:
        print(f"Failed to retrieve IP address for camera {CAMERA_ID}.")

if __name__ == "__main__":
    main()


ERROR:__main__:Error retrieving camera IP: 401 Client Error: Unauthorized for url: http://192.168.20.1/api/rest/v1/cameras/dfebfa67-3a9d-4e79-aa1b-2789dec1d844


Failed to retrieve IP address for camera dfebfa67-3a9d-4e79-aa1b-2789dec1d844.


In [14]:
import requests
import websocket

import json

# Define the API endpoint
url = 'ws://192.168.20.1/api/ws/events/v1'  # Replace with the actual URL

# Manually set your access token
access_token = 'eyJhbGciOiJSUzI1NiIsImtpZCI6IkIxMUNDMDE5RjQ2MEE4OTBFMkVGQTQ2RkMwOEM2QjI0IiwidHlwIjoiSldUIn0.eyJpc3MiOiJodHRwOi8vc3J2LW1pbGVzdG9uZS5hZC5heG9uZS9JRFAiLCJuYmYiOjE3MzE0MDkxMDcsImlhdCI6MTczMTQwOTEwNywiZXhwIjoxNzMxNDEyNzA3LCJhdWQiOlsibWFuYWdlbWVudHNlcnZlciIsImh0dHA6Ly9zcnYtbWlsZXN0b25lLmFkLmF4b25lL0lEUC9yZXNvdXJjZXMiXSwic2NvcGUiOlsibWFuYWdlbWVudHNlcnZlciJdLCJhbXIiOlsicHdkIl0sImNsaWVudF9pZCI6IkdyYW50VmFsaWRhdG9yQ2xpZW50Iiwic3ViIjoiMTM4RkZFRkMtNEExRC00RUY3LTk3NDYtNDYzN0ZEMTRDQkIxIiwiYXV0aF90aW1lIjoxNzMxNDA5MTA3LCJpZHAiOiJsb2NhbCIsIm5hbWUiOiJBeG9uZSIsInVwZGF0ZWRfYXQiOiIxNzMwOTEwNTQyIiwianRpIjoiOUMxNEI0QkNERUExN0RGQUI3RkIxQUVENjI4OEZFMEMifQ.2I_xeFNymvSBSMO2kSjwhUncJ9T-E9wJLWA2hMcN2DjVl2U6k8_YBQ05b7ABy37fTyxV50j_Q3GG88OHUkXWbHhA5ZEuIJVVrIHM1XkpQt8DTy1NxQTiKg3vvwio0mEFsYNj8pjr4Q0m9Nx81OShtwlgH7npeUSRfVthhPLvZ_70Cw8iXdOtG8lsg5GpFr36qvM_PGhRVCDgmYEWVDsPMh9cr34O7p_NujuuRxwndL76DOjbprV0S1t0yZ591gYdJYyqjdklgxdv96CMek6ITLW4enHORKiLymwhfhb8793XjBUythsurDHZ8G98AT8kB-ByRTCOtGE6DM5MtQIoFA'  # Replace with your access token

# Define a callback function to handle incoming messages from the WebSocket
def on_message(ws, message):
    print("Received message:")
    print(message)

    # Optionally, you can save the incoming message to a file
    with open("events_data.json", "a") as file:
        json.dump(json.loads(message), file, indent=4)
        file.write("\n")  # Write each message on a new line

def on_error(ws, error):
    print("Error:", error)

def on_close(ws, close_status_code, close_msg):
    print("Closed connection")

def on_open(ws):
    print("Opened WebSocket connection")
    # Optionally, send a message if the WebSocket connection requires a handshake or request
    # ws.send('{"type": "some_action"}')

# Create and configure the WebSocket client
headers = {
    'Authorization': f'Bearer {access_token}',
}

# Initiating WebSocket connection
ws = websocket.WebSocketApp(
    url,
    header=headers,
    on_message=on_message,
    on_error=on_error,
    on_close=on_close,
)

# Run the WebSocket client
ws.run_forever()

INFO:websocket:Websocket connected
INFO:websocket:tearing down on exception 


Error: 
Closed connection


True

In [ ]:
import os
import requests
import logging
import json
from requests.exceptions import RequestException

# Configuration
USERNAME = "axone"
PASSWORD = "Ax0nesys!"
SERVER_URL = "http://192.168.20.1"
VERIFY_CERTIFICATES = os.getenv('VERIFY_CERTIFICATES', 'True') == 'True'
RECORDING_SERVER_ID = "27ed6e15-babf-4c4f-a86e-cf3d2a188ec1"  # Provided recording server ID

# Logging setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def get_access_token(session):
    """Retrieve access token from the Identity Provider (IDP)."""
    try:
        response = session.post(
            f"{SERVER_URL}/IDP/connect/token",
            headers={'Content-Type': 'application/x-www-form-urlencoded'},
            data={
                'grant_type': 'password',
                'username': USERNAME,
                'password': PASSWORD,
                'client_id': 'GrantValidatorClient'
            },
            verify=VERIFY_CERTIFICATES
        )
        response.raise_for_status()
        token = response.json().get('access_token')
        if not token:
            logger.error("Access token not found in the response.")
            return None
        logger.info("Access token retrieved successfully.")

        # Save the token to a text file
        with open('access_token.txt', 'w') as file:
            file.write(token)
        logger.info("Access token saved to 'access_token.txt'.")
        
        return token
    except RequestException as e:
        logger.error(f"Error retrieving token: {e}")
        return None

def fetch_and_save_data(endpoint, filename, token):
    """Fetch data from the API and save it as a JSON file."""
    try:
        url = f"{SERVER_URL}{endpoint}"
        headers = {"Authorization": f"Bearer {token}"}
        response = requests.get(url, headers=headers, verify=VERIFY_CERTIFICATES)
        response.raise_for_status()
        data = response.json()

        # Create the data directory if it does not exist
        if not os.path.exists("data"):
            os.makedirs("data")

        # Save the data to a JSON file
        with open(f"data/{filename}.json", "w") as file:
            json.dump(data, file, indent=4)
        logger.info(f"Data saved to 'data/{filename}.json'")
    except RequestException as e:
        logger.error(f"Error fetching data from {endpoint}: {e}")

def main():
    with requests.Session() as session:
        token = get_access_token(session)
        if not token:
            logger.error("Failed to retrieve access token. Exiting.")
            return

        # Fetch data from various endpoints
        fetch_and_save_data("/api/rest/v1/events", "events", token)
        fetch_and_save_data("/api/rest/v1/eventTypes", "event_types", token)
        fetch_and_save_data("/api/rest/v1/alarms", "alarms", token)
        fetch_and_save_data("/api/rest/v1/recordingServers", "recording_servers", token)
        fetch_and_save_data("/api/rest/v1/system", "system_info", token)
        fetch_and_save_data("/api/rest/v1/system/health", "system_health", token)

        # Fetch and save storage information for the given recording server
        storage_endpoint = f"/api/rest/v1/recordingServers/{RECORDING_SERVER_ID}/storage"
        fetch_and_save_data(storage_endpoint, "recording_server_storage", token)

        logger.info("All data has been retrieved and saved successfully.")

if __name__ == "__main__":
    main()


INFO:__main__:Access token retrieved successfully.
INFO:__main__:Access token saved to 'access_token.txt'.
INFO:__main__:Data saved to 'data/events.json'
INFO:__main__:Data saved to 'data/event_types.json'
INFO:__main__:Data saved to 'data/alarms.json'
INFO:__main__:Data saved to 'data/recording_servers.json'
ERROR:__main__:Error fetching data from /api/rest/v1/system: 404 Client Error: Not Found for url: http://192.168.20.1/api/rest/v1/system
ERROR:__main__:Error fetching data from /api/rest/v1/system/health: 404 Client Error: Not Found for url: http://192.168.20.1/api/rest/v1/system/health
ERROR:__main__:Error fetching data from /api/rest/v1/recordingServers/27ed6e15-babf-4c4f-a86e-cf3d2a188ec1/storage: 404 Client Error: Not Found for url: http://192.168.20.1/api/rest/v1/recordingServers/27ed6e15-babf-4c4f-a86e-cf3d2a188ec1/storage
INFO:__main__:All data has been retrieved and saved successfully.


In [1]:
import os
import requests
import logging
import json
from requests.exceptions import RequestException

# Configuration
USERNAME = "axone"
PASSWORD = "Ax0nesys!"
SERVER_URL = "http://192.168.20.1"
VERIFY_CERTIFICATES = os.getenv('VERIFY_CERTIFICATES', 'True') == 'True'
RECORDING_SERVER_ID = "27ed6e15-babf-4c4f-a86e-cf3d2a188ec1"  # Provided recording server ID

# Logging setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def get_access_token(session):
    """Retrieve access token from the Identity Provider (IDP)."""
    try:
        response = session.post(
            f"{SERVER_URL}/IDP/connect/token",
            headers={'Content-Type': 'application/x-www-form-urlencoded'},
            data={
                'grant_type': 'password',
                'username': USERNAME,
                'password': PASSWORD,
                'client_id': 'GrantValidatorClient'
            },
            verify=VERIFY_CERTIFICATES
        )
        response.raise_for_status()
        token = response.json().get('access_token')
        if not token:
            logger.error("Access token not found in the response.")
            return None
        logger.info("Access token retrieved successfully.")

        # Save the token to a text file
        with open('access_token.txt', 'w') as file:
            file.write(token)
        logger.info("Access token saved to 'access_token.txt'.")
        
        return token
    except RequestException as e:
        logger.error(f"Error retrieving token: {e}")
        return None

def fetch_and_save_data(endpoint, filename, token):
    """Fetch data from the API and save it as a JSON file."""
    try:
        url = f"{SERVER_URL}{endpoint}"
        headers = {"Authorization": f"Bearer {token}"}
        response = requests.get(url, headers=headers, verify=VERIFY_CERTIFICATES)
        response.raise_for_status()
        data = response.json()

        # Create the data directory if it does not exist
        if not os.path.exists("data"):
            os.makedirs("data")

        # Save the data to a JSON file
        with open(f"data/{filename}.json", "w") as file:
            json.dump(data, file, indent=4)
        logger.info(f"Data saved to 'data/{filename}.json'")
    except RequestException as e:
        logger.error(f"Error fetching data from {endpoint}: {e}")

def main():
    with requests.Session() as session:
        token = get_access_token(session)
        if not token:
            logger.error("Failed to retrieve access token. Exiting.")
            return

        # Fetch data from various endpoints
        fetch_and_save_data("/api/rest/v1/events", "events", token)
        fetch_and_save_data("/api/rest/v1/eventTypes", "event_types", token)
        fetch_and_save_data("/api/rest/v1/alarms", "alarms", token)
        fetch_and_save_data("/api/rest/v1/recordingServers", "recording_servers", token)
        fetch_and_save_data("/api/rest/v1/system", "system_info", token)
        fetch_and_save_data("/api/rest/v1/system/health", "system_health", token)

        # Fetch and save storage information for the given recording server
        storage_endpoint = f"/api/rest/v1/recordingServers/{RECORDING_SERVER_ID}/storage"
        fetch_and_save_data(storage_endpoint, "recording_server_storage", token)

        logger.info("All data has been retrieved and saved successfully.")

if __name__ == "__main__":
    main()


INFO:__main__:Access token retrieved successfully.
INFO:__main__:Access token saved to 'access_token.txt'.
INFO:__main__:Data saved to 'data/events.json'
INFO:__main__:Data saved to 'data/event_types.json'
INFO:__main__:Data saved to 'data/alarms.json'
INFO:__main__:Data saved to 'data/recording_servers.json'
ERROR:__main__:Error fetching data from /api/rest/v1/system: 404 Client Error: Not Found for url: http://192.168.20.1/api/rest/v1/system
ERROR:__main__:Error fetching data from /api/rest/v1/system/health: 404 Client Error: Not Found for url: http://192.168.20.1/api/rest/v1/system/health
ERROR:__main__:Error fetching data from /api/rest/v1/recordingServers/27ed6e15-babf-4c4f-a86e-cf3d2a188ec1/storage: 404 Client Error: Not Found for url: http://192.168.20.1/api/rest/v1/recordingServers/27ed6e15-babf-4c4f-a86e-cf3d2a188ec1/storage
INFO:__main__:All data has been retrieved and saved successfully.


In [14]:
import websocket
import json
import threading

# Replace with your API gateway, token, and settings
API_GATEWAY = "ws://192.168.20.1/api/ws/events/v1"
ACCESS_TOKEN = "eyJhbGciOiJSUzI1NiIsImtpZCI6IkIxMUNDMDE5RjQ2MEE4OTBFMkVGQTQ2RkMwOEM2QjI0IiwidHlwIjoiSldUIn0.eyJpc3MiOiJodHRwOi8vc3J2LW1pbGVzdG9uZS5hZC5heG9uZS9JRFAiLCJuYmYiOjE3MzIxMDgxOTAsImlhdCI6MTczMjEwODE5MCwiZXhwIjoxNzMyMTExNzkwLCJhdWQiOlsibWFuYWdlbWVudHNlcnZlciIsImh0dHA6Ly9zcnYtbWlsZXN0b25lLmFkLmF4b25lL0lEUC9yZXNvdXJjZXMiXSwic2NvcGUiOlsibWFuYWdlbWVudHNlcnZlciJdLCJhbXIiOlsicHdkIl0sImNsaWVudF9pZCI6IkdyYW50VmFsaWRhdG9yQ2xpZW50Iiwic3ViIjoiMTM4RkZFRkMtNEExRC00RUY3LTk3NDYtNDYzN0ZEMTRDQkIxIiwiYXV0aF90aW1lIjoxNzMyMTA4MTkwLCJpZHAiOiJsb2NhbCIsIm5hbWUiOiJBeG9uZSIsInVwZGF0ZWRfYXQiOiIxNzMyMDk2NDc3IiwianRpIjoiQ0M2QTA3QjA3NzQzNDc2NDIwMTMxMUQzOEVEREQ2QTIifQ.I9E9TCpjYe60N5dry8Q_4sT4KXrV17wQtKEaHOTmaYvoSpEqN8LNuE4ksct5V1lf5xnl-HteRD4SmeiylofRKZ6aA8Yx-6azoNpYnWhdLaoF7wSAunZFjr3pAosOZOjIij-irLaxp0o2z_WMpdEymQT8PdyQUqmy9RxKZlo8D1nhfg14dCwo8HHnx2PACJigF1HR2sy54pwR-H1YngSSIeXWErNSygnM89yaG7uRzV1zwFBqbncTYmpGaxa3zQLZje59SVW6h9mWux5JuxzjpAjIRcMBIb77kDsgDyHBUUrgmrLncDDitjBCOfFv1zizHytm55hq3gh67GiOQZ4Img"  # Replace with your bearer token

def on_message(ws, message):
    """Handles incoming messages from the WebSocket."""
    data = json.loads(message)
    print(f"Received data: {json.dumps(data, indent=2)}")

def on_error(ws, error):
    """Handles errors."""
    print(f"WebSocket error: {error}")

def on_close(ws, close_status_code, close_msg):
    """Handles the closing of the WebSocket connection."""
    print(f"WebSocket closed with status: {close_status_code}, message: {close_msg}")

def on_open(ws):
    """Handles actions on WebSocket connection open."""
    print("WebSocket connection established.")

    # Start a session
    start_session_message = {
        "command": "startSession",
        "data": {}
    }
    ws.send(json.dumps(start_session_message))
    print("Session started.")

    # Subscribe to all events
    subscribe_message = {
        "command": "subscribe",
        "data": {
            "eventFilter": {
                "eventType": "*",  # Subscribe to all event types
                "source": "*"      # Subscribe to all sources
            }
        }
    }
    ws.send(json.dumps(subscribe_message))
    print("Subscribed to all events.")

def run_websocket():
    """Runs the WebSocket client."""
    headers = [
        f"Authorization: Bearer {ACCESS_TOKEN}"
    ]
    ws = websocket.WebSocketApp(
        API_GATEWAY,
        header=headers,
        on_open=on_open,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close
    )
    ws.run_forever()

if __name__ == "__main__":
    print("Starting WebSocket client...")
    websocket_thread = threading.Thread(target=run_websocket)
    websocket_thread.start()


Starting WebSocket client...


ERROR:websocket:Handshake status 500 Internal Server Error -+-+- {'content-length': '2772', 'server': 'Microsoft-HTTPAPI/2.0', 'x-powered-by': 'ASP.NET', 'date': 'Wed, 20 Nov 2024 13:12:39 GMT'} -+-+- b"Unexpected error validating client. System.ServiceModel.Security.MessageSecurityException: Une faute non s\xc3\xa9curis\xc3\xa9e ou incorrectement s\xc3\xa9curis\xc3\xa9e a \xc3\xa9t\xc3\xa9 re\xc3\xa7ue de l'autre partie. Voir le FaultException interne pour le code et les d\xc3\xa9tails de la faute. ---> System.ServiceModel.FaultException: Le jeton du contexte de s\xc3\xa9curit\xc3\xa9 a expir\xc3\xa9 ou n'est pas valide. Le message n'a pas \xc3\xa9t\xc3\xa9 trait\xc3\xa9.\r\n   --- Fin de la trace de la pile d'exception interne ---\r\n\r\nServer stack trace: \r\n   \xc3\xa0 System.ServiceModel.Channels.SecurityChannelFactory`1.SecurityRequestChannel.ProcessReply(Message reply, SecurityProtocolCorrelationState correlationState, TimeSpan timeout)\r\n   \xc3\xa0 System.ServiceModel.Chann

WebSocket error: Handshake status 500 Internal Server Error -+-+- {'content-length': '2772', 'server': 'Microsoft-HTTPAPI/2.0', 'x-powered-by': 'ASP.NET', 'date': 'Wed, 20 Nov 2024 13:12:39 GMT'} -+-+- b"Unexpected error validating client. System.ServiceModel.Security.MessageSecurityException: Une faute non s\xc3\xa9curis\xc3\xa9e ou incorrectement s\xc3\xa9curis\xc3\xa9e a \xc3\xa9t\xc3\xa9 re\xc3\xa7ue de l'autre partie. Voir le FaultException interne pour le code et les d\xc3\xa9tails de la faute. ---> System.ServiceModel.FaultException: Le jeton du contexte de s\xc3\xa9curit\xc3\xa9 a expir\xc3\xa9 ou n'est pas valide. Le message n'a pas \xc3\xa9t\xc3\xa9 trait\xc3\xa9.\r\n   --- Fin de la trace de la pile d'exception interne ---\r\n\r\nServer stack trace: \r\n   \xc3\xa0 System.ServiceModel.Channels.SecurityChannelFactory`1.SecurityRequestChannel.ProcessReply(Message reply, SecurityProtocolCorrelationState correlationState, TimeSpan timeout)\r\n   \xc3\xa0 System.ServiceModel.Chan